# 1. What the raw data is actually like

Before any analysis: what is in this dataset, what is missing from it, and which
columns cannot be read at face value.

The source is 1,224 Claude Code session logs from a single user's machine,
recorded between June and July 2026. Everything below comes from the anonymized
extract, not from the raw logs.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

DATA = Path.cwd().parent / "data"
sessions = pd.read_parquet(DATA / "sessions.parquet")
events = pd.read_parquet(DATA / "events.parquet")
print(f"{len(sessions):,} sessions, {len(events):,} events")

1,213 sessions, 181,406 events


## Coverage

Sessions are not evenly sized. Most are short, a handful run for thousands of
events, and the mean is therefore not a useful summary of anything here.

In [2]:
from agent_telemetry.analysis import sessions as session_analysis

session_analysis.session_overview(sessions)

,metric,value
0,sessions,1213.0
1,events,181406.0
2,median events per session,33.0
3,median duration (minutes),2.4
4,total tokens (millions),17840.8
5,tool calls,45574.0
6,tool errors,1186.0
7,subagent sessions,1017.0
8,main sessions,196.0
9,distinct projects,15.0


In [3]:
session_analysis.length_distribution(sessions)

,metric,mean,p50,p75,p90,p95,p99
0,events,149.6,33.0,113.0,223.0,349.2,2535.7
1,duration_minutes,41.6,2.4,3.4,9.7,39.1,616.0
2,tool_calls,37.6,10.0,41.0,79.8,110.4,406.3
3,total_tokens,14707996.3,1671516.0,8032397.0,16420763.8,25083016.8,366175926.2


The distance between the mean and the median is the story: the mean session
has several times as many events as the median one. Every later chart uses
medians for this reason.

## Missing values

Empty strings rather than nulls, because the extractor writes a category or an
empty string and never a missing value. This is where to look before believing
any grouped result.

In [4]:
blank = (sessions == "").sum()[lambda s: s > 0].sort_values(ascending=False).to_frame("blank rows")
blank["share"] = (blank["blank rows"] / len(sessions)).round(3)
blank

,blank rows,share
efforts,1105,0.911
branch,208,0.171
top_tool,96,0.079
models,54,0.045
primary_model,54,0.045
started_at,1,0.001
ended_at,1,0.001
client_version,1,0.001


`primary_model` is blank where a session produced no assistant message at all:
an aborted start, or a session that only ever changed a setting. Those rows are
real sessions and are kept, but they have to be filtered out of any model
comparison.

## Columns that mean less than they appear to

**`duration_minutes`** is the wall clock distance between the first and last
event. A session left open overnight reports the whole night. It is a measure of
elapsed time, not of work.

In [5]:
long = sessions.nlargest(5, "duration_minutes")[
    ["duration_minutes", "events", "tool_calls", "primary_model"]
]
long["minutes_per_event"] = (long["duration_minutes"] / long["events"]).round(2)
long

,duration_minutes,events,tool_calls,primary_model,minutes_per_event
1144,13020.2,2460,442,other,5.29
1098,9510.8,915,157,deepseek-v4-flash,10.39
65,5382.5,2940,661,claude-sonnet-5,1.83
81,1550.0,3163,487,claude-opus-4-8,0.49
87,1494.1,664,118,deepseek-v4-flash,2.25


The sessions with the longest durations are not the ones with the most events.
Any analysis that treats duration as effort is measuring how long a terminal
window stayed open.

**`total_tokens`** counts cache reads, and a cache read re-counts the whole
context on every single turn. The number is real but it is not comparable to a
token count from a one-shot API call.

In [6]:
from agent_telemetry.analysis import economics

mix = economics.token_mix(sessions).to_frame("tokens")
mix["share"] = (mix["tokens"] / mix["tokens"].sum()).round(4)
mix

,tokens,share
input_tokens,170227671,0.0095
output_tokens,46618902,0.0026
cache_creation_tokens,144665026,0.0081
cache_read_tokens,17479287945,0.9797


## Schema drift

The client updated repeatedly across the period. If a field changed meaning, it
shows up here first.

In [7]:
session_analysis.version_timeline(sessions).tail(12)

,client_version,sessions,median_events,mean_cache_hit_rate
3,2.1.197,20,39.0,0.7819
4,2.1.198,7,107.0,0.8657
5,2.1.201,9,121.0,0.9088
6,2.1.206,7,174.0,0.9409
7,2.1.207,19,317.0,0.9129
8,2.1.211,14,49.0,0.7511
9,2.1.212,37,32.0,0.4970
10,2.1.214,15,34.0,0.5061
11,2.1.215,51,14.0,0.3064
12,2.1.216,8,151.5,0.8675


Cache hit rate per version is stable enough that the token accounting can be
treated as consistent across the whole window.